# DDPM 모델 4종 사용 노트북

이 노트북은 현재 코드베이스에 구현된 4개 모델을 같은 방식으로 실행할 수 있게 묶은 실행 템플릿입니다.

1. Base Conditional DDPM
2. Guided Conditional DDPM (classifier-free + classifier)
3. Loop-Conditioned DDPM
4. Unconditional DDPM


In [ ]:
from __future__ import annotations

import json
import sys
from dataclasses import asdict
from datetime import datetime
from pathlib import Path
from pprint import pprint

import torch

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "src").exists() and (REPO_ROOT.parent / "src").exists():
    REPO_ROOT = REPO_ROOT.parent

src_path = str(REPO_ROOT / "src")
if src_path not in sys.path:
    sys.path.insert(0, src_path)

print("repo:", REPO_ROOT)


In [ ]:
from diffusion_hash_inv.models.conditional_diffusion import (
    ConditionalDiffusionTrainConfig,
    cleanup_torch_resources,
    train_conditional_diffusion,
)
from diffusion_hash_inv.models.guided_conditional_diffusion import (
    GuidedConditionalDiffusionTrainConfig,
    train_guided_conditional_diffusion,
)
from diffusion_hash_inv.models.loop_conditioned_diffusion import (
    LoopConditionedDiffusionTrainConfig,
    loop_state_keys,
    timestep_to_state_indices,
    train_loop_conditioned_diffusion,
)
from diffusion_hash_inv.models.unconditional_ddpm import (
    UnconditionalDDPMTrainConfig,
    train_unconditional_ddpm,
)


## 공통 경로와 설정

`data/images/<run-id>/message.png`와 `output/json/**/<run-id>.json`이 이미 생성되어 있어야 합니다.


In [ ]:
# ─── 모드 선택: "smoke" 또는 "full" ──────────────────────────────────────────
MODE = "full"   # "full" 로 변경하면 아래 full training 설정이 적용됩니다.

DATA_ROOT       = REPO_ROOT / "data" / "images"
JSON_ROOT       = REPO_ROOT / "output" / "json"
NOTEBOOK_OUTPUT = REPO_ROOT / "output" / f"diffusion_model_{MODE}_{datetime.now().strftime('%Y%m%d_%H%M%S')}"

# ─── smoke run 설정 (빠른 동작 확인용) ──────────────────────────────────────
_SMOKE = dict(
    device        = "cpu",
    max_images    = 16,
    image_size    = 32,
    batch_size    = 2,
    train_steps   = 2,
    timesteps     = 2,
    base_channels = 4,
    time_dim      = 8,
    sample_count  = 1,
)

# ─── full training 설정 ─────────────────────────────────────────────────────
_FULL = dict(
    device        = "mps",    # Apple Silicon. CUDA 환경: "cuda"
    max_images    = None,     # 전체 데이터셋 사용
    image_size    = 64,
    batch_size    = 64,
    train_steps   = 1_000,
    timesteps     = 1_000,
    base_channels = 64,
    time_dim      = 256,
    sample_count  = 4,
)

# ─── 활성 설정 적용 ─────────────────────────────────────────────────────────
_cfg = _SMOKE if MODE == "smoke" else _FULL
DEVICE        = _cfg["device"]
MAX_IMAGES    = _cfg["max_images"]
IMAGE_SIZE    = _cfg["image_size"]
BATCH_SIZE    = _cfg["batch_size"]
TRAIN_STEPS   = _cfg["train_steps"]
TIMESTEPS     = _cfg["timesteps"]
BASE_CHANNELS = _cfg["base_channels"]
TIME_DIM      = _cfg["time_dim"]
SAMPLE_COUNT  = _cfg["sample_count"]


# ─── 이미지 전처리 방식 ─────────────────────────────────────────────────────
# "resize"        : image_size×image_size로 bilinear 축소 (메모리 효율, 비율 왜곡)
# "reshape"       : 픽셀 순서 재배치 → 112×112 (정보 손실 없음, image_size 무시)
# "height-flatten": 28×28 타일 16개를 4×4 그리드로 재배치 → 112×112 (구조 보존)
FIT_MODE = "height-flatten"

# ─── 프로세스트레ース設定 (포워드/리버스 각 timestep 이미지 저장) ──────────────
# True로 설정하면 output_dir/process_traces/{forward,reverse}/t_XXXXXX.png 저장
# full 모드에서 timesteps=1056 → 각 2112개 PNG / run. 디스크 용량 주의.
SAVE_PROCESS_TRACES = True
_TRACE_SAMPLE_COUNT = dict(smoke=1, full=1)
TRACE_SAMPLE_COUNT = _TRACE_SAMPLE_COUNT[MODE]

print(f"MODE={MODE}  device={DEVICE}  images={MAX_IMAGES}  size={IMAGE_SIZE}  "
    f"steps={TRAIN_STEPS}  timesteps={TIMESTEPS}  ch={BASE_CHANNELS}  tdim={TIME_DIM}")
print("data exists:", DATA_ROOT.exists())
print("json exists:", JSON_ROOT.exists())
print("message.png count:", len(list(DATA_ROOT.rglob("message.png"))) if DATA_ROOT.exists() else 0)


In [ ]:
# ─── 실행 플래그 ────────────────────────────────────────────────────────────
# 실행할 모델의 플래그를 True로 변경하세요.
RUN_BASE_CONDITIONAL = False
RUN_GUIDED           = False   # classifier-free + classifier 둘 다 실행
RUN_LOOP_CONDITIONED = True
RUN_UNCONDITIONAL    = False
RUN_ALL              = False   # 위 플래그 무시하고 15개 job 순차 실행


In [ ]:
def show_config(config):
    payload = asdict(config)
    for key in ("data_root", "json_root", "output_dir", "beta_values_path"):
        if key in payload and payload[key] is not None:
            payload[key] = str(payload[key])
    pprint(payload)

def show_result(result):
    printable = {k: str(v) if isinstance(v, Path) else v for k, v in result.items()}
    pprint(printable)
    sample = result.get("sample_grid")
    if sample is not None:
        print("sample exists:", Path(sample).exists(), sample)

def mem_cleanup(*names: str) -> None:
    """지정한 변수를 globals에서 제거하고 torch 캐시를 비웁니다."""
    g = globals()
    for name in names:
        g.pop(name, None)
    cleanup_torch_resources()
    print("memory cleaned")


## 1. Base Conditional DDPM

Base conditional DDPM은 `message.png`를 입력 이미지로 사용하고, condition label은 final hash만 사용합니다.


In [ ]:
_base_common = dict(
    data_root=DATA_ROOT,
    json_root=JSON_ROOT,
    image_size=IMAGE_SIZE,
    fit_mode=FIT_MODE,
    label_source="final-hash",
    max_images=MAX_IMAGES,
    batch_size=BATCH_SIZE,
    train_steps=TRAIN_STEPS,
    base_channels=BASE_CHANNELS,
    time_dim=TIME_DIM,
    sample_count=SAMPLE_COUNT,
    device=DEVICE,
    save_process_traces=SAVE_PROCESS_TRACES,
    trace_sample_count=TRACE_SAMPLE_COUNT,
)

# Approach 1/2는 beta 길이가 해시 통계에서 결정되므로 timesteps="auto" 사용
base_linear_config = ConditionalDiffusionTrainConfig(
    **_base_common,
    output_dir=NOTEBOOK_OUTPUT / "01a_base_linear",
    timesteps="auto",
    beta_schedule="linear",
)
base_approach1_config = ConditionalDiffusionTrainConfig(
    **_base_common,
    output_dir=NOTEBOOK_OUTPUT / "01b_base_approach1",
    timesteps="auto",
    beta_schedule="hash-approach1",
)
base_approach2_config = ConditionalDiffusionTrainConfig(
    **_base_common,
    output_dir=NOTEBOOK_OUTPUT / "01c_base_approach2",
    timesteps="auto",
    beta_schedule="hash-approach2",
)

print("--- linear ---")
show_config(base_linear_config)
print("--- hash-approach1 ---")
show_config(base_approach1_config)
print("--- hash-approach2 ---")
show_config(base_approach2_config)


In [ ]:
if RUN_BASE_CONDITIONAL and not RUN_ALL:
    print("\n=== linear ===")
    base_linear_result = train_conditional_diffusion(base_linear_config)
    show_result(base_linear_result)

    print("\n=== hash-approach1 ===")
    base_approach1_result = train_conditional_diffusion(base_approach1_config)
    show_result(base_approach1_result)

    print("\n=== hash-approach2 ===")
    base_approach2_result = train_conditional_diffusion(base_approach2_config)
    show_result(base_approach2_result)
elif RUN_ALL:
    print("RUN_ALL이 True이므로 개별 실행을 건너뜁니다.")
else:
    print("Set RUN_BASE_CONDITIONAL = True to train the base conditional DDPM.")


In [ ]:
if not RUN_ALL:
    mem_cleanup(
        "base_linear_result", "base_approach1_result", "base_approach2_result",
        "base_linear_config", "base_approach1_config", "base_approach2_config",
    )


## 2. Guided Conditional DDPM

`classifier-free`와 `classifier` 두 가지 guidance mode를 순서대로 실행합니다.


In [ ]:
_guided_cfg_common = dict(
    data_root=DATA_ROOT,
    json_root=JSON_ROOT,
    image_size=IMAGE_SIZE,
    fit_mode=FIT_MODE,
    label_source="final-hash",
    max_images=MAX_IMAGES,
    batch_size=BATCH_SIZE,
    train_steps=TRAIN_STEPS,
    base_channels=BASE_CHANNELS,
    time_dim=TIME_DIM,
    classifier_base_channels=BASE_CHANNELS,
    guidance_mode="classifier-free",
    guidance_scale=2.0,
    condition_dropout=0.1,
    sample_count=SAMPLE_COUNT,
    device=DEVICE,
    save_process_traces=SAVE_PROCESS_TRACES,
    trace_sample_count=TRACE_SAMPLE_COUNT,
)

_guided_cls_common = dict(
    data_root=DATA_ROOT,
    json_root=JSON_ROOT,
    image_size=IMAGE_SIZE,
    fit_mode=FIT_MODE,
    label_source="final-hash",
    max_images=MAX_IMAGES,
    batch_size=BATCH_SIZE,
    train_steps=TRAIN_STEPS,
    base_channels=BASE_CHANNELS,
    time_dim=TIME_DIM,
    classifier_base_channels=BASE_CHANNELS,
    guidance_mode="classifier",
    guidance_scale=1.0,
    condition_dropout=0.0,
    sample_count=SAMPLE_COUNT,
    device=DEVICE,
    save_process_traces=SAVE_PROCESS_TRACES,
    trace_sample_count=TRACE_SAMPLE_COUNT,
)

guided_cfg_linear_config = GuidedConditionalDiffusionTrainConfig(
    **_guided_cfg_common,
    output_dir=NOTEBOOK_OUTPUT / "02a_guided_cfg_linear",
    timesteps="auto",
    beta_schedule="linear",
)
guided_cfg_approach1_config = GuidedConditionalDiffusionTrainConfig(
    **_guided_cfg_common,
    output_dir=NOTEBOOK_OUTPUT / "02b_guided_cfg_approach1",
    timesteps="auto",
    beta_schedule="hash-approach1",
)
guided_cfg_approach2_config = GuidedConditionalDiffusionTrainConfig(
    **_guided_cfg_common,
    output_dir=NOTEBOOK_OUTPUT / "02c_guided_cfg_approach2",
    timesteps="auto",
    beta_schedule="hash-approach2",
)

guided_cls_linear_config = GuidedConditionalDiffusionTrainConfig(
    **_guided_cls_common,
    output_dir=NOTEBOOK_OUTPUT / "02d_guided_cls_linear",
    timesteps="auto",
    beta_schedule="linear",
)
guided_cls_approach1_config = GuidedConditionalDiffusionTrainConfig(
    **_guided_cls_common,
    output_dir=NOTEBOOK_OUTPUT / "02e_guided_cls_approach1",
    timesteps="auto",
    beta_schedule="hash-approach1",
)
guided_cls_approach2_config = GuidedConditionalDiffusionTrainConfig(
    **_guided_cls_common,
    output_dir=NOTEBOOK_OUTPUT / "02f_guided_cls_approach2",
    timesteps="auto",
    beta_schedule="hash-approach2",
)

print("--- classifier-free ---")
show_config(guided_cfg_linear_config)
show_config(guided_cfg_approach1_config)
show_config(guided_cfg_approach2_config)
print("--- classifier ---")
show_config(guided_cls_linear_config)
show_config(guided_cls_approach1_config)
show_config(guided_cls_approach2_config)


In [ ]:
if RUN_GUIDED and not RUN_ALL:
    print("\n=== classifier-free (linear) ===")
    guided_cfg_linear_result = train_guided_conditional_diffusion(guided_cfg_linear_config)
    show_result(guided_cfg_linear_result)
    cleanup_torch_resources()

    print("\n=== classifier-free (hash-approach1) ===")
    guided_cfg_approach1_result = train_guided_conditional_diffusion(guided_cfg_approach1_config)
    show_result(guided_cfg_approach1_result)
    cleanup_torch_resources()

    print("\n=== classifier-free (hash-approach2) ===")
    guided_cfg_approach2_result = train_guided_conditional_diffusion(guided_cfg_approach2_config)
    show_result(guided_cfg_approach2_result)
    cleanup_torch_resources()

    print("\n=== classifier (linear) ===")
    guided_cls_linear_result = train_guided_conditional_diffusion(guided_cls_linear_config)
    show_result(guided_cls_linear_result)
    cleanup_torch_resources()

    print("\n=== classifier (hash-approach1) ===")
    guided_cls_approach1_result = train_guided_conditional_diffusion(guided_cls_approach1_config)
    show_result(guided_cls_approach1_result)
    cleanup_torch_resources()

    print("\n=== classifier (hash-approach2) ===")
    guided_cls_approach2_result = train_guided_conditional_diffusion(guided_cls_approach2_config)
    show_result(guided_cls_approach2_result)
elif RUN_ALL:
    print("RUN_ALL이 True이므로 개별 실행을 건너뜁니다.")
else:
    print("Set RUN_GUIDED = True to train the guided conditional DDPM.")


In [ ]:
if not RUN_ALL:
    mem_cleanup(
        "guided_cfg_linear_result", "guided_cfg_approach1_result", "guided_cfg_approach2_result",
        "guided_cls_linear_result", "guided_cls_approach1_result", "guided_cls_approach2_result",
        "guided_cfg_linear_config", "guided_cfg_approach1_config", "guided_cfg_approach2_config",
        "guided_cls_linear_config", "guided_cls_approach1_config", "guided_cls_approach2_config",
    )


## 3. Loop-Conditioned DDPM

`Logs/4th Step/1st Round`의 `Loop Start + 64개 Loop + Loop End`에서 `A/B/C/D` 값을 읽어 `(66, 4)` temporal condition tensor로 사용합니다. 1056-step hash-derived beta schedule에서는 각 state가 16개 diffusion timestep에 대응합니다.


In [ ]:
_loop_common = dict(
    data_root=DATA_ROOT,
    json_root=JSON_ROOT,
    image_size=IMAGE_SIZE,
    fit_mode=FIT_MODE,
    max_images=MAX_IMAGES,
    batch_size=BATCH_SIZE,
    train_steps=TRAIN_STEPS,
    base_channels=BASE_CHANNELS,
    time_dim=TIME_DIM,
    condition_step="4th Step",
    condition_round="1st Round",
    loop_count=64,
    sample_count=SAMPLE_COUNT,
    device=DEVICE,
    save_process_traces=SAVE_PROCESS_TRACES,
    trace_sample_count=TRACE_SAMPLE_COUNT,
)

loop_linear_config = LoopConditionedDiffusionTrainConfig(
    **_loop_common,
    output_dir=NOTEBOOK_OUTPUT / "03a_loop_linear",
    timesteps="auto",
    beta_schedule="linear",
)
loop_approach1_config = LoopConditionedDiffusionTrainConfig(
    **_loop_common,
    output_dir=NOTEBOOK_OUTPUT / "03b_loop_approach1",
    timesteps="auto",
    beta_schedule="hash-approach1",
)
loop_approach2_config = LoopConditionedDiffusionTrainConfig(
    **_loop_common,
    output_dir=NOTEBOOK_OUTPUT / "03c_loop_approach2",
    timesteps="auto",
    beta_schedule="hash-approach2",
)

_loop_state_keys = loop_state_keys(loop_linear_config.loop_count)
_loop_timesteps_per_state = len(loop_linear_config.word_names) * 4
_loop_hash_timesteps = len(_loop_state_keys) * _loop_timesteps_per_state
_loop_probe = torch.tensor([
    0,
    _loop_timesteps_per_state - 1,
    _loop_timesteps_per_state,
    _loop_hash_timesteps - 1,
])
_loop_probe_indices = timestep_to_state_indices(
    _loop_probe,
    state_count=len(_loop_state_keys),
    words_per_state=len(loop_linear_config.word_names),
    diffusion_timesteps=_loop_hash_timesteps,
).tolist()
print(
    f"loop temporal states: {len(_loop_state_keys)} "
    f"({_loop_state_keys[0]} -> ... -> {_loop_state_keys[-1]})"
)
print(
    f"hash-aligned timesteps: {_loop_hash_timesteps} = "
    f"{len(_loop_state_keys)} states * {_loop_timesteps_per_state} byte steps/state"
)
print(f"mapping check t={_loop_probe.tolist()} -> state index={_loop_probe_indices}")

print("--- linear ---")
show_config(loop_linear_config)
print("--- hash-approach1 ---")
show_config(loop_approach1_config)
print("--- hash-approach2 ---")
show_config(loop_approach2_config)


In [ ]:
if RUN_LOOP_CONDITIONED and not RUN_ALL:
    print("\n=== linear ===")
    loop_linear_result = train_loop_conditioned_diffusion(loop_linear_config)
    show_result(loop_linear_result)
    cleanup_torch_resources()

    print("\n=== hash-approach1 ===")
    loop_approach1_result = train_loop_conditioned_diffusion(loop_approach1_config)
    show_result(loop_approach1_result)
    cleanup_torch_resources()

    print("\n=== hash-approach2 ===")
    loop_approach2_result = train_loop_conditioned_diffusion(loop_approach2_config)
    show_result(loop_approach2_result)
elif RUN_ALL:
    print("RUN_ALL이 True이므로 개별 실행을 건너뜁니다.")
else:
    print("Set RUN_LOOP_CONDITIONED = True to train the loop-conditioned DDPM.")


In [ ]:
if not RUN_ALL:
    mem_cleanup(
        "loop_linear_result", "loop_approach1_result", "loop_approach2_result",
        "loop_linear_config", "loop_approach1_config", "loop_approach2_config",
    )


## 4. Unconditional DDPM

Unconditional DDPM은 condition이나 JSON label을 사용하지 않고 `message.png` 이미지만으로 학습합니다.


In [ ]:
_uncond_common = dict(
    data_root=DATA_ROOT,
    json_root=JSON_ROOT,
    image_size=IMAGE_SIZE,
    fit_mode=FIT_MODE,
    max_images=MAX_IMAGES,
    batch_size=BATCH_SIZE,
    train_steps=TRAIN_STEPS,
    base_channels=BASE_CHANNELS,
    time_dim=TIME_DIM,
    sample_count=SAMPLE_COUNT,
    device=DEVICE,
    save_process_traces=SAVE_PROCESS_TRACES,
    trace_sample_count=TRACE_SAMPLE_COUNT,
)

unconditional_linear_config = UnconditionalDDPMTrainConfig(
    **_uncond_common,
    output_dir=NOTEBOOK_OUTPUT / "04a_unconditional_linear",
    timesteps="auto",
    beta_schedule="linear",
)
unconditional_approach1_config = UnconditionalDDPMTrainConfig(
    **_uncond_common,
    output_dir=NOTEBOOK_OUTPUT / "04b_unconditional_approach1",
    timesteps="auto",
    beta_schedule="hash-approach1",
)
unconditional_approach2_config = UnconditionalDDPMTrainConfig(
    **_uncond_common,
    output_dir=NOTEBOOK_OUTPUT / "04c_unconditional_approach2",
    timesteps="auto",
    beta_schedule="hash-approach2",
)

print("--- linear ---")
show_config(unconditional_linear_config)
print("--- hash-approach1 ---")
show_config(unconditional_approach1_config)
print("--- hash-approach2 ---")
show_config(unconditional_approach2_config)


In [ ]:
if RUN_UNCONDITIONAL and not RUN_ALL:
    print("\n=== linear ===")
    unconditional_linear_result = train_unconditional_ddpm(unconditional_linear_config)
    show_result(unconditional_linear_result)
    cleanup_torch_resources()

    print("\n=== hash-approach1 ===")
    unconditional_approach1_result = train_unconditional_ddpm(unconditional_approach1_config)
    show_result(unconditional_approach1_result)
    cleanup_torch_resources()

    print("\n=== hash-approach2 ===")
    unconditional_approach2_result = train_unconditional_ddpm(unconditional_approach2_config)
    show_result(unconditional_approach2_result)
elif RUN_ALL:
    print("RUN_ALL이 True이므로 개별 실행을 건너뜁니다.")
else:
    print("Set RUN_UNCONDITIONAL = True to train the unconditional DDPM.")


In [ ]:
if not RUN_ALL:
    mem_cleanup(
        "unconditional_linear_result", "unconditional_approach1_result", "unconditional_approach2_result",
        "unconditional_linear_config", "unconditional_approach1_config", "unconditional_approach2_config",
    )


## 전체 실행

`RUN_ALL = True`로 설정하면 5개 모델을 순서대로 실행합니다. `MODE` 설정에 따라 smoke 또는 full training으로 동작합니다.


In [ ]:
if RUN_ALL:
    all_results = {}

    print("[1/15] base_conditional (linear)")
    all_results["base_linear"] = train_conditional_diffusion(base_linear_config)
    show_result(all_results["base_linear"])
    cleanup_torch_resources()

    print("[2/15] base_conditional (hash-approach1)")
    all_results["base_approach1"] = train_conditional_diffusion(base_approach1_config)
    show_result(all_results["base_approach1"])
    cleanup_torch_resources()

    print("[3/15] base_conditional (hash-approach2)")
    all_results["base_approach2"] = train_conditional_diffusion(base_approach2_config)
    show_result(all_results["base_approach2"])
    cleanup_torch_resources()

    print("[4/15] guided classifier-free (linear)")
    all_results["guided_cfg_linear"] = train_guided_conditional_diffusion(guided_cfg_linear_config)
    show_result(all_results["guided_cfg_linear"])
    cleanup_torch_resources()

    print("[5/15] guided classifier-free (hash-approach1)")
    all_results["guided_cfg_approach1"] = train_guided_conditional_diffusion(guided_cfg_approach1_config)
    show_result(all_results["guided_cfg_approach1"])
    cleanup_torch_resources()

    print("[6/15] guided classifier-free (hash-approach2)")
    all_results["guided_cfg_approach2"] = train_guided_conditional_diffusion(guided_cfg_approach2_config)
    show_result(all_results["guided_cfg_approach2"])
    cleanup_torch_resources()

    print("[7/15] guided classifier (linear)")
    all_results["guided_cls_linear"] = train_guided_conditional_diffusion(guided_cls_linear_config)
    show_result(all_results["guided_cls_linear"])
    cleanup_torch_resources()

    print("[8/15] guided classifier (hash-approach1)")
    all_results["guided_cls_approach1"] = train_guided_conditional_diffusion(guided_cls_approach1_config)
    show_result(all_results["guided_cls_approach1"])
    cleanup_torch_resources()

    print("[9/15] guided classifier (hash-approach2)")
    all_results["guided_cls_approach2"] = train_guided_conditional_diffusion(guided_cls_approach2_config)
    show_result(all_results["guided_cls_approach2"])
    cleanup_torch_resources()

    print("[10/15] loop_conditioned (linear)")
    all_results["loop_linear"] = train_loop_conditioned_diffusion(loop_linear_config)
    show_result(all_results["loop_linear"])
    cleanup_torch_resources()

    print("[11/15] loop_conditioned (hash-approach1)")
    all_results["loop_approach1"] = train_loop_conditioned_diffusion(loop_approach1_config)
    show_result(all_results["loop_approach1"])
    cleanup_torch_resources()

    print("[12/15] loop_conditioned (hash-approach2)")
    all_results["loop_approach2"] = train_loop_conditioned_diffusion(loop_approach2_config)
    show_result(all_results["loop_approach2"])
    cleanup_torch_resources()

    print("[13/15] unconditional (linear)")
    all_results["uncond_linear"] = train_unconditional_ddpm(unconditional_linear_config)
    show_result(all_results["uncond_linear"])
    cleanup_torch_resources()

    print("[14/15] unconditional (hash-approach1)")
    all_results["uncond_approach1"] = train_unconditional_ddpm(unconditional_approach1_config)
    show_result(all_results["uncond_approach1"])
    cleanup_torch_resources()

    print("[15/15] unconditional (hash-approach2)")
    all_results["uncond_approach2"] = train_unconditional_ddpm(unconditional_approach2_config)
    show_result(all_results["uncond_approach2"])
    cleanup_torch_resources()

    print("\n=== 전체 결과 ===")
    for name, result in all_results.items():
        print(f"  {name}: loss={result['final_loss']:.4f}")
else:
    print("Set RUN_ALL = True to run all fifteen models. MODE 설정에 따라 smoke 또는 full training으로 실행됩니다.")


In [ ]:
if RUN_ALL:
    mem_cleanup(
        "all_results",
        "base_linear_config", "base_approach1_config", "base_approach2_config",
        "guided_cfg_linear_config", "guided_cfg_approach1_config", "guided_cfg_approach2_config",
        "guided_cls_linear_config", "guided_cls_approach1_config", "guided_cls_approach2_config",
        "loop_linear_config", "loop_approach1_config", "loop_approach2_config",
        "unconditional_linear_config", "unconditional_approach1_config", "unconditional_approach2_config",
    )
